# Introduction to pandas

1. Creating a DataFrame
2. Select columns & rows from a DataFrame
3. Calculate aggregations/descriptive statistics
4. Grouped calculations

## The Basics

In [ ]:
import pandas as pd

### Create a pandas DataFrame

In [ ]:
d = {'a': 1, 'b': 2, 'c': 3}
d['b']

In [ ]:
df = pd.DataFrame({
    'eye':    ['brown', 'green', 'blue', 'blue', 'brown', 'brown'],
    'hair':   ['brown', 'blonde', 'red', 'blonde', 'brown', 'blue'],
    'height': [60,       72,      62,     65,      80,      74    ],
})

# print(df) # Jupyter will display the last line of your cell!
df

### Selecting Columns & Rows

In [ ]:
df.dtypes

In [ ]:
# df['height'] # select single column
df[['height', 'eye']] # select multiple columns

#### Comparators
- Less than `<`
- Greater than `>`
- Less than/equal to `<=`
- Greater than/equal to `>=`
- Is equal to `==`
- Not equal to `!=`

In [ ]:
# df['eye'] == 'brown'
# df['height'] == 60
# df['height'] <= 70

# df.loc[df['height'] <= 70] # filtering on a boolean criteria
df.loc[df['eye'] == 'brown'] # individuals with brown eyes

In [ ]:
df

In [ ]:
(df['eye'] == 'brown') & (df['height'] <= 70) # combine boolean expressions
# df.loc[(df['eye'] == 'brown') & (df['height'] <= 70)]

(df['eye'] == 'brown') | (df['height'] <= 70) # combine boolean expressions
# df.loc[(df['eye'] == 'brown') | (df['height'] <= 70)]

df.loc[(df['eye'] != 'brown')]

### Calculations
- algebraic operations
    - addition, subtraction, multiplication, division, power
- aggregations
    - min, max
    - mean, median, mode
    - standard deviation

In [ ]:
df.dtypes

In [ ]:
# df.select_dtypes('object')
# ((df['height'] + 10).astype(str)) + ' inches'
# df['height'] + 100.1

# df['height'] - 100
# df['height'] / 10
# df['height'] * 100
# df['height'] ** 2

# df['height'].max()
# df['height'].min()
# df['height'].mean()
# df['height'].median()
# df['height'].quantile([.25, .5, .75])
df['height'].std()

In [ ]:
df.groupby('eye')['height'].mean()

## Applied to Fred
### Load Data from Fred

In [ ]:
from fredapi import Fred
fred = Fred()

# OECD Total share price index 
oecd_series = (
    fred.get_series(
        'SPASTT01USM661N', observation_start='1960-01-01', observation_end='2023-07-01'
    )
)

cpi_series =(
    fred.get_series('CPIAUCSL', observation_start='1960-01-01', observation_end='2023-07-01')
)

fred_df = pd.DataFrame({
    'OECD': oecd_series,
    'CPI': cpi_series
})

fred_df.head(5)
# data.head() # view first N rows/entries
# data.tail() # view last N rows/entries

In [ ]:
# fred_df['CPI_adj'] = 1 # Make a new column in pandas
fred_df['CPI_adj'] = fred_df['CPI'] / fred_df.loc['2023-01-01']['CPI']
fred_df

## Load Data from a File (.csv)
- [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [`pandas.read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)

In [ ]:
!pwd

In [ ]:
df_gop = pd.read_csv(
    "../../materials/presidential-term-data/GOP_flag.csv",
    dtype={'Date': str, 'GOP': bool}
)

df_gop = df_gop.drop_duplicates() # remove duplicate rows from data

# Seeing party is easier to understand than boolean of GOP
df_gop['party'] = df_gop['GOP'].map({True: 'GOP', False: 'Dem'})

# Dates are currently strings and we need them to be datetimes
# we are updating the date column with the converted version of itself
# Note that .str.ljust is a quick fix for improper dates
df_gop['Date'] = pd.to_datetime(
    df_gop['Date'].str.ljust(7, '0'), format='%Y.%m'
)

df_gop.head()

## Simple Econometrics

First we need to combine ALL of our data. We covered constructing a NEW DataFrame given some parts, but we can also combine 2 DataFrames along a specified column via [`.merge`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html). If you have used Excel, this is essentially a VLOOKUP:


In [ ]:
from numpy import log

final_df = fred_df.merge(df_gop, left_index=True, right_on='Date')
final_df = final_df.loc[final_df['Date'] >= '1970'] # let's only consider dates later than 2007

final_df['real_total_return'] = final_df['OECD'] * final_df['CPI_adj']
final_df['log_real_total_return'] = log(final_df['real_total_return'])
final_df['party_lag'] = final_df['party'].shift()

final_df.head()

In [ ]:
# Ordinary Least Squares Regression
## Statsmodels has 2 ways of conducting regressions
### - from a formula
### - from design matrices
## We'll focus on computing a regression from a formula
from statsmodels.formula.api import ols

# model formulation https://www.statsmodels.org/stable/examples/notebooks/generated/formulas.html
#            dependent   ~ independent
#            endogenous  ~ exodenous
#                      y ~ x
model = ols('log_real_total_return ~ party_lag', data=final_df) # create linear model
fit = model.fit()                                  # run/fit the model

# Note that Y is contiuous/numeric (CPI_adj_log)
#      and X is categorical (party)

fit.summary() # print out a summary of the regression; 
             # note that this did not perform well

In [ ]:
from numpy import array
from numpy.random import default_rng
from matplotlib.pyplot import subplots

fig, (ax1, ax2) = subplots(
    1, 2, figsize=(10, 3), gridspec_kw={'width_ratios': [2, 1]}, sharey=True
)
order = {'Dem': 0, 'GOP': 1}
colors = {'Dem': 'tab:blue', 'GOP': 'tab:red'}

## First plot → focus on the context of the question at hand
ax1.fill_between(
    final_df['Date'], final_df['log_real_total_return'], where=final_df['party_lag'] == 'GOP',
    color=colors['GOP'], interpolate='pre',
)
ax1.fill_between(
    final_df['Date'], final_df['log_real_total_return'], where=final_df['party_lag'] == 'Dem',
    color=colors['Dem'], interpolate='pre',
)

ax1.set_ylabel('log(Real Total Return)')

## Second plot → focus on the categorical linear regression
rng = default_rng(0)
for party, group in final_df.groupby('party_lag'):
    ax2.scatter(
        order[party] + rng.normal(scale=.05, size=len(group)), 
        group['log_real_total_return'],
        s=10, alpha=.2,
        color=colors[party]
    )
ax2.set_xticks([*order.values()], labels=order.keys())

x = array([0, 1])
y = (x * fit.params.loc["party_lag[T.GOP]"]) + fit.params.loc['Intercept']
ax2.plot(x, y, color='k', lw=2)
ax2.yaxis.set_tick_params(left=True, labelleft=True)
ax1.set_title(
    'Statistically Significant Effect of Party on Real Total Returns', 
    size='x-large', loc='left',
    pad=20
)